# 01 — EDA, Qualidade de Dados e Reestruturação PEDE 2025
Análise exploratória longitudinal da base Passos Mágicos (2022–2025), auditoria de valores ausentes, verificação de esquema e leitura analítica da reestruturação metodológica.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import settings
from src.pipeline.data_validation import validate_cohort_contract
from src.pipeline.features import standardize_year_sheet

xl = pd.ExcelFile(settings.RAW_DATA_PATH)
frames = []
validation_report = {}
for yr in [2022, 2023, 2024, 2025]:
    raw_yr = pd.read_excel(xl, f"PEDE{yr}")
    std_yr = standardize_year_sheet(raw_yr, yr)
    rep_yr = validate_cohort_contract(std_yr, yr)
    validation_report[yr] = rep_yr
    frames.append(std_yr)

df_raw = pd.concat(frames, ignore_index=True)
df_raw["ano_referencia"] = df_raw["ano"]
print(f"Registros totais: {len(df_raw)} | Colunas: {df_raw.shape[1]}")
print("Anos presentes:", sorted(df_raw["ano_referencia"].dropna().unique().tolist()))
for yr, rep in validation_report.items():
    print(f"Contrato {yr}: passed={rep.passed} | rows={rep.total_rows} | unique_ras={rep.unique_ras} | checks={rep.checks}")


Registros totais: 4092 | Colunas: 25
Anos presentes: [2022, 2023, 2024, 2025]
Contrato 2022: passed=True | rows=860 | unique_ras=860 | checks={'schema_columns_present': True, 'ra_uniqueness': True, 'indicators_plausible_range': True, 'completeness_sla': True}
Contrato 2023: passed=True | rows=1014 | unique_ras=1014 | checks={'schema_columns_present': True, 'ra_uniqueness': True, 'indicators_plausible_range': True, 'completeness_sla': True}
Contrato 2024: passed=True | rows=1156 | unique_ras=1156 | checks={'schema_columns_present': True, 'ra_uniqueness': True, 'indicators_plausible_range': True, 'completeness_sla': True}
Contrato 2025: passed=True | rows=1062 | unique_ras=1062 | checks={'schema_columns_present': True, 'ra_uniqueness': True, 'indicators_plausible_range': True, 'completeness_sla': True}


## 1. Cobertura Temporal e Volume por Ciclo
Verificação da distribuição de estudantes/registos por ano de referência para avaliar continuidade longitudinal e possíveis quebras de série.

In [2]:
cobertura_ano = (
    df_raw.groupby("ano_referencia", dropna=False)
    .size()
    .reset_index(name="total_registros")
    .assign(percentual=lambda d: (d["total_registros"] / len(df_raw) * 100).round(2))
)
display(cobertura_ano)

,ano_referencia,total_registros,percentual
0,2022,860,21.02
1,2023,1014,24.78
2,2024,1156,28.25
3,2025,1062,25.95


## 2. Auditoria de Valores Ausentes (Top 15 Colunas)
Identificação de lacunas de preenchimento que exigem tratamento específico ou cautela na engenharia de features.

In [3]:
nulos_df = (
    pd.DataFrame({
        "qtd_nulos": df_raw.isna().sum(),
        "pct_nulos": (df_raw.isna().mean() * 100).round(2),
        "dtype": df_raw.dtypes.astype(str),
    })
    .sort_values(["pct_nulos", "qtd_nulos"], ascending=False)
)
display(nulos_df.head(15))

,qtd_nulos,pct_nulos,dtype
ing,2525,61.71,float64
inde_anterior,2048,50.05,float64
pedra_anterior_ord,1586,38.76,float64
IPP,1038,25.37,float64
idade,399,9.75,float64
por,215,5.25,float64
mat,214,5.23,float64
INDE,185,4.52,float64
pedra_ord,185,4.52,float64
IDA,178,4.35,float64


## 3. Estatísticas Descritivas Numéricas e Contexto PEDE 2025
Resumo quantitativo das variáveis numéricas e registro das implicações da reestruturação metodológica observada em 2025 para o desenho anti-leakage ( 
t → t+1).

In [4]:
num_cols = df_raw.select_dtypes(include="number").columns.tolist()
if num_cols:
    display(df_raw[num_cols].describe().T[["count", "mean", "std", "min", "50%", "max"]].round(3))
else:
    print("Nenhuma coluna numérica detectada diretamente.")

,count,mean,std,min,50%,max
ano,4092.0,2023.591,1.087,2022.00,2024.000,2025.000
fase_num,4092.0,2.679,2.277,0.00,2.000,9.000
idade,3693.0,12.441,3.155,7.00,12.000,27.000
ano_ingresso,4092.0,2022.012,1.891,2016.00,2022.000,2025.000
anos_na_ong,4092.0,1.579,1.583,0.00,1.000,7.000
defasagem,4092.0,-0.690,0.966,-8.00,-1.000,3.000
INDE,3907.0,7.300,1.033,2.69,7.410,9.760
IAA,3927.0,8.116,2.368,0.00,8.800,10.002
IEG,4016.0,8.122,2.008,0.00,8.700,10.000
IPS,3921.0,6.418,1.882,2.50,7.500,10.000
